In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import micromlgen

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Path to the folder containing your 8 CSV files
DATA_DIR = '/content/drive/MyDrive/dataset/mmdata12526/Final Individual DataSet'
MODEL_EXPORT_FILE = 'RUPAM_model.h'

# These are the columns we want to extract from the sensors.
# We ignore datetime, wifi, time_sync, etc.
FEATURE_COLS = [
    'rd_distance_mm', 'rd_velocity_cm_s', 'rd_angle_deg',
    'rd_x_mm', 'rd_y_mm', 'c4001_distance_m',
    'c4001_velocity_m_s', 'c4001_energy'
]

def main():
    print(">>> Starting ESP32 Sensor ML Pipeline <<<")

    # ==========================================
    # 2. DATA LABELING & PREPROCESSING
    # ==========================================
    all_data = []
    csv_files = glob.glob(os.path.join(DATA_DIR, '*.csv'))

    if not csv_files:
        print("No CSV files found in the specified directory.")
        return

    for file in csv_files:
        # Extract the class name from the file name (e.g., 'SITTING.csv' -> 'SITTING')
        class_name = os.path.basename(file).replace('.csv', '')

        try:
            df = pd.read_csv(file)

            # Ensure all required feature columns are present (skip missing ones if necessary)
            available_cols = [c for c in FEATURE_COLS if c in df.columns]
            df = df[available_cols]

            # Drop any rows that had NaN values from sensor glitches
            df = df.dropna()

            # ==========================================
            # 3. NOISE REDUCTION & FEATURE EXTRACTION
            # ==========================================
            # Apply Exponential Moving Average (EMA).
            # EMA is incredibly efficient for C++ microcontrollers.
            # Formula: smoothed = (alpha * current_val) + ((1 - alpha) * previous_smoothed)
            alpha = 0.3
            df[available_cols] = df[available_cols].ewm(alpha=alpha, adjust=False).mean()

            # Add Label
            df['label'] = class_name
            all_data.append(df)
            print(f"Loaded and preprocessed {len(df)} samples for class: {class_name}")

        except Exception as e:
            print(f"Error processing {file}: {e}")

    # Combine all classes into one DataFrame
    dataset = pd.concat(all_data, ignore_index=True)

    # Separate features (X) and labels (y)
    X = dataset[FEATURE_COLS].values
    y_raw = dataset['label'].values

    # Encode string labels into integers (0 to 7)
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)
    class_mapping = {int(k): str(v) for k, v in enumerate(label_encoder.classes_)}

    print("\nClass Mapping:")
    for k, v in class_mapping.items():
        print(f"{k} -> {v}")

    # ==========================================
    # 4. SMOTE (DATA BALANCING)
    # ==========================================
    print("\nApplying SMOTE to balance the classes...")
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)
    print(f"Original dataset shape: {X.shape}")
    print(f"Resampled dataset shape: {X_resampled.shape}")

    # ==========================================
    # 5. TRAIN-TEST SPLIT
    # ==========================================
    X_train, X_test, y_train, y_test = train_test_split(
        X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
    )

    # ==========================================
    # 6. MODEL TRAINING & OPTIMIZATION
    # ==========================================
    print("\nStarting Model Training & Hyperparameter Optimization...")
    # >>> HIGHLY RESTRICTED FOR ESP32 FLASH MEMORY <<<
    # Drastically reduced estimators and depth to keep the exported C++ file small.
    param_grid = {
        'n_estimators': [5, 10, 15],  # Drastically reduced from 20-50
        'max_depth': [3, 5, 7],       # Shallower trees to prevent exponential if-else growth
        'min_samples_split': [5, 10], # Require more samples to split (keeps trees smaller)
        'min_samples_leaf': [2, 4]
    }

    base_rf = RandomForestClassifier(random_state=42)

    # Randomized Search for optimization
    rf_optimized = RandomizedSearchCV(
        estimator=base_rf,
        param_distributions=param_grid,
        n_iter=10,
        cv=3,
        scoring='accuracy',
        random_state=42,
        n_jobs=-1
    )

    rf_optimized.fit(X_train, y_train)
    best_model = rf_optimized.best_estimator_

    print(f"Best Hyperparameters for ESP32 deployment: {rf_optimized.best_params_}")

    # ==========================================
    # 7. MODEL EVALUATION
    # ==========================================
    print("\nEvaluating Model on Test Data...")
    y_pred = best_model.predict(X_test)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=[class_mapping[i] for i in range(len(class_mapping))]))

    # ==========================================
    # 8. MODEL EXPORT FOR ESP32S3
    # ==========================================
    print("\nExporting Model to C++ header file...")
    try:
        # micromlgen ports the sklearn model to pure C++
        c_code = micromlgen.port(best_model, classmap=class_mapping)
        with open(MODEL_EXPORT_FILE, 'w') as f:
            f.write(c_code)
        print(f"Success! Model exported to '{MODEL_EXPORT_FILE}'.")
        print(f"File size approximation: {os.path.getsize(MODEL_EXPORT_FILE) / 1024:.2f} KB")
        print("Include this file in your Arduino/PlatformIO project.")
    except Exception as e:
        print(f"Error exporting model: {e}")
        print("Make sure micromlgen is installed: pip install micromlgen")

if __name__ == "__main__":
    main()

>>> Starting ESP32 Sensor ML Pipeline <<<
Loaded and preprocessed 17759 samples for class: WALKING
Loaded and preprocessed 17398 samples for class: STANDING
Loaded and preprocessed 18597 samples for class: SITTING
Loaded and preprocessed 6208 samples for class: SIT TO STAND
Loaded and preprocessed 12966 samples for class: EXIT
Loaded and preprocessed 15000 samples for class: NO PERSON
Loaded and preprocessed 12346 samples for class: ENTRY
Loaded and preprocessed 5232 samples for class: STAND TO SIT

Class Mapping:
0 -> ENTRY
1 -> EXIT
2 -> NO PERSON
3 -> SIT TO STAND
4 -> SITTING
5 -> STAND TO SIT
6 -> STANDING
7 -> WALKING

Applying SMOTE to balance the classes...
Original dataset shape: (105506, 8)
Resampled dataset shape: (148776, 8)

Starting Model Training & Hyperparameter Optimization...
Best Hyperparameters for ESP32 deployment: {'n_estimators': 15, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 7}

Evaluating Model on Test Data...
Accuracy: 0.8565

Classification R